# NUS ST4248 — Statistical Learning II
## Complete intuition-first tutorial notebook

This notebook turns the major ST4248 ideas into a sequence of **small experiments + real-data case studies**.

### What you will learn

1. Statistical-learning setup and bias–variance
2. Polynomial regression
3. Regression splines
4. Smoothing splines
5. Kernel regression / Nadaraya–Watson estimator
6. LOESS / local regression
7. Additive models
8. Regression and classification trees
9. Gini impurity and entropy
10. Tree complexity and cost-complexity pruning
11. Bagging
12. Bootstrap 63.2% phenomenon
13. Random forests and feature subsampling
14. Out-of-bag validation
15. Feature importance and permutation importance
16. Gradient boosting
17. Linear support-vector classifiers
18. Soft margins and the role of $C$
19. Kernel SVMs and the RBF kernel
20. Neural networks, gradient training and $L_2$ regularisation
21. Cross-validation and hyperparameter tuning
22. Unified model comparison
23. Global vs local vs partition-based vs compositional modelling

The recurring questions are:

> **What structure does the model assume?**  
> **What controls flexibility?**  
> **How does that affect bias and variance?**  
> **How do we estimate generalisation honestly?**

## 0. Why three datasets?

We use three complementary settings.

### A. Controlled nonlinear one-dimensional data

Here we know the true function $f(x)$. This lets us directly see underfitting, overfitting and smoothing.

### B. Diabetes regression dataset

A real multivariate regression problem from scikit-learn. It is small enough for classroom experimentation and supports linear, spline, tree, ensemble and neural-network comparisons.

### C. Breast Cancer Wisconsin dataset

A real binary classification problem. It is useful for trees, impurity criteria, random forests, boosting, SVMs, ROC curves and neural networks.

All datasets are bundled with scikit-learn, so the notebook is reproducible without Kaggle credentials or a network download.

In [45]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.interpolate import UnivariateSpline
from statsmodels.nonparametric.kernel_regression import KernelReg
from statsmodels.nonparametric.smoothers_lowess import lowess

from sklearn.base import clone
from sklearn.datasets import load_diabetes, load_breast_cancer, make_blobs
from sklearn.ensemble import (
    BaggingRegressor,
    RandomForestRegressor,
    RandomForestClassifier,
    GradientBoostingRegressor,
    GradientBoostingClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_validate, GridSearchCV,
)
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

from bokeh.io import output_notebook, show
from bokeh.layouts import row
from bokeh.models import ColumnDataSource, DataTable, TableColumn
from bokeh.plotting import figure

from IPython.display import display, HTML
import html


output_notebook()

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
print("Setup complete.")

Loading BokehJS ...

Setup complete.


### Display helpers

The table helper intentionally converts pandas column names to strings.  
This prevents a common Bokeh `ColumnDataSource` failure when a DataFrame contains integer-valued column names.

In [2]:


def show_table(
    df,
    title="Table",
    width=950,
    height=260,
    digits=4
):
    """
    Stable Jupyter table renderer.

    Unlike Bokeh DataTable, this generates ordinary HTML,
    so table rows remain visible after saving/reopening
    the notebook.
    """

    x = df.copy()

    # ------------------------------------------------------------------
    # 1. Normalise column names
    # ------------------------------------------------------------------
    x.columns = [str(c) for c in x.columns]

    # ------------------------------------------------------------------
    # 2. Round numeric columns
    # ------------------------------------------------------------------
    numeric_cols = x.select_dtypes(include=np.number).columns

    for c in numeric_cols:
        x[c] = x[c].round(digits)

    x = x.reset_index(drop=True)

    # ------------------------------------------------------------------
    # 3. Convert dataframe to persistent HTML
    # ------------------------------------------------------------------
    table_html = x.to_html(
        index=False,
        border=0,
        classes="st4248-table",
        justify="left"
    )

    # ------------------------------------------------------------------
    # 4. Give it notebook-friendly styling
    # ------------------------------------------------------------------
    full_html = f"""
    <div style="
        width:{width}px;
        max-width:100%;
        margin:8px 0 18px 0;
        font-family:Arial, Helvetica, sans-serif;
    ">

        <div style="
            font-size:15px;
            font-weight:600;
            margin-bottom:8px;
        ">
            {html.escape(str(title))}
        </div>

        <div style="
            max-height:{height}px;
            overflow:auto;
            border:1px solid #dddddd;
            border-radius:5px;
        ">

            <style>

                .st4248-table {{
                    width:100%;
                    border-collapse:collapse;
                    font-size:13px;
                }}

                .st4248-table thead th {{
                    position:sticky;
                    top:0;
                    background:#f5f5f5;
                    font-weight:600;
                    text-align:left;
                    padding:8px 10px;
                    border-bottom:2px solid #cccccc;
                }}

                .st4248-table tbody td {{
                    padding:7px 10px;
                    border-bottom:1px solid #eeeeee;
                    text-align:left;
                }}

                .st4248-table tbody tr:nth-child(even) {{
                    background:#fafafa;
                }}

                .st4248-table tbody tr:hover {{
                    background:#f0f0f0;
                }}

            </style>

            {table_html}

        </div>

    </div>
    """

    display(HTML(full_html))

def reg_metrics(y_true, pred):
    return {
        "RMSE": mean_squared_error(y_true, pred) ** 0.5,
        "R2": r2_score(y_true, pred),
    }

def cls_metrics(y_true, pred, prob):
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "ROC_AUC": roc_auc_score(y_true, prob),
    }

# Part I — Statistical learning as function estimation

Assume

$$
Y=f(X)+\varepsilon.
$$

The goal is to estimate the unknown $f$.

A rigid model has fewer possible shapes and usually lower variance.  
A flexible model can approximate more complicated signals but can also react strongly to noise.

A useful decomposition is

$$
E[(Y-\hat f(X))^2]
=
\sigma^2
+
\operatorname{Bias}[\hat f(X)]^2
+
\operatorname{Var}[\hat f(X)].
$$

The irreducible error $\sigma^2$ cannot be eliminated by model complexity.

So the central modelling problem becomes:

$$
\boxed{
\text{enough flexibility to learn structure}
\quad\text{but not enough to learn noise}
}
$$

In [3]:
def f_true(x):
    return np.sin(1.35*x) + 0.25*x + 0.35*np.cos(2.6*x)

n = 90
x = np.sort(rng.uniform(-3.2, 3.2, n))
y = f_true(x) + rng.normal(0, 0.35, n)

grid = np.linspace(x.min(), x.max(), 400)
truth_grid = f_true(grid)

p = figure(width=900, height=430, title="Controlled nonlinear regression problem",
           x_axis_label="x", y_axis_label="y")
p.scatter(x, y, size=7, alpha=0.6, legend_label="observations")
p.line(grid, truth_grid, line_width=3, legend_label="true f(x)")
p.legend.location = "top_left"
show(p)

# Part II — Polynomial regression

Polynomial regression replaces the single feature $x$ by a basis:

$$
x,\;x^2,\;x^3,\ldots,x^d.
$$

Then

$$
f(x)=\beta_0+\beta_1x+\cdots+\beta_dx^d.
$$

The model is nonlinear in $x$ but still linear in the coefficients.

### Complexity knob

The degree $d$ controls flexibility.

- small $d$: smooth and restrictive,
- moderate $d$: can capture curvature,
- very large $d$: can become unstable, especially near boundaries.

The experiment below deliberately includes an excessive degree.

In [44]:
X1 = x.reshape(-1, 1)
G1 = grid.reshape(-1, 1)

p = figure(width=950, height=470,
           title="Polynomial regression: increasing degree increases flexibility",
           x_axis_label="x", y_axis_label="y")
p.scatter(x, y, size=6, alpha=0.35, legend_label="observations")
p.line(grid, truth_grid, line_width=4, alpha=0.45, legend_label="true f(x)")

for degree in [1, 3, 7, 15]:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("ridge", Ridge(alpha=1e-6)),
    ])
    model.fit(X1, y)
    p.line(grid, model.predict(G1), line_width=2.5, legend_label=f"degree={degree}")

p.legend.location = "top_left"
p.legend.click_policy = "hide"
show(p)

### Student inference

Do not conclude that high-degree polynomials are “bad.”  
The real lesson is broader:

$$
\text{more expressive function class}
\Rightarrow
\text{potentially lower approximation bias}
\Rightarrow
\text{potentially higher sampling variance}.
$$

That same pattern will reappear for trees, kernel smoothers, SVMs and neural networks.

# Part III — Regression splines

A spline avoids fitting one global high-degree polynomial.

Instead, it uses polynomial pieces connected smoothly at **knots**.

A basis representation is

$$
f(x)=\sum_{j=1}^{m}\beta_j b_j(x),
$$

where the $b_j(x)$ are spline basis functions.

### Intuition

A local change in the curve need not force the entire function to move dramatically.

### Complexity knobs

- number/location of knots,
- polynomial degree,
- coefficient regularisation.

In [5]:
p = figure(width=950, height=470,
           title="Cubic regression splines with different knot counts",
           x_axis_label="x", y_axis_label="y")
p.scatter(x, y, size=6, alpha=0.35, legend_label="observations")
p.line(grid, truth_grid, line_width=4, alpha=0.45, legend_label="true f(x)")

for knots in [4, 7, 12]:
    model = Pipeline([
        ("spline", SplineTransformer(n_knots=knots, degree=3, include_bias=False)),
        ("ridge", Ridge(alpha=1e-3)),
    ])
    model.fit(X1, y)
    p.line(grid, model.predict(G1), line_width=2.5, legend_label=f"{knots} knots")

p.legend.location = "top_left"
p.legend.click_policy = "hide"
show(p)

# Part IV — Smoothing splines

Smoothing splines make the fit-versus-complexity competition explicit:

$$
\min_f
\left[
\sum_{i=1}^{n}(y_i-f(x_i))^2
+
\lambda\int [f''(t)]^2dt
\right].
$$

The first term rewards fit.

The curvature penalty

$$
\int [f''(t)]^2dt
$$

discourages excessive bending.

SciPy's `UnivariateSpline` uses a smoothing parameter `s`.  
Its numerical parameterisation differs from the textbook $\lambda$, but the intuition is the same: stronger smoothing allows larger residual error in exchange for a smoother function.

In [6]:
p = figure(width=950, height=470,
           title="Smoothing splines: fit versus smoothness",
           x_axis_label="x", y_axis_label="y")
p.scatter(x, y, size=6, alpha=0.35, legend_label="observations")
p.line(grid, truth_grid, line_width=4, alpha=0.45, legend_label="true f(x)")

for s in [0.5, 8, 35]:
    sm = UnivariateSpline(x, y, s=s)
    p.line(grid, sm(grid), line_width=2.5, legend_label=f"s={s}")

p.legend.location = "top_left"
p.legend.click_policy = "hide"
show(p)

# Part V — Kernel regression / Nadaraya–Watson estimator

Kernel regression predicts at $x_0$ using a weighted average of nearby outcomes:

$$
\hat f(x_0)
=
\frac{\sum_i K_h(x_i-x_0)y_i}
{\sum_i K_h(x_i-x_0)}.
$$

For a Gaussian kernel,

$$
K_h(u)\propto \exp\left(-\frac{u^2}{2h^2}\right).
$$

### The bandwidth $h$

- small $h$ → very local, highly flexible,
- large $h$ → broad neighbourhood, smoother fit.

Bandwidth is therefore a direct complexity parameter.

In [7]:
p = figure(width=950, height=470,
           title="Nadaraya–Watson kernel regression: bandwidth controls locality",
           x_axis_label="x", y_axis_label="y")
p.scatter(x, y, size=6, alpha=0.35, legend_label="observations")
p.line(grid, truth_grid, line_width=4, alpha=0.45, legend_label="true f(x)")

for h in [0.12, 0.35, 0.9]:
    kr = KernelReg(endog=y, exog=x, var_type="c", reg_type="lc", bw=[h])
    pred, _ = kr.fit(grid)
    p.line(grid, pred, line_width=2.5, legend_label=f"bandwidth={h}")

p.legend.location = "top_left"
p.legend.click_policy = "hide"
show(p)

### Inspecting one prediction

At one target location $x_0$, the normalised weights are

$$
w_i=
\frac{K_h(x_i-x_0)}
{\sum_j K_h(x_j-x_0)}.
$$

Then

$$
\hat f(x_0)=\sum_iw_i y_i.
$$

The next table shows which observations dominate one local prediction.

In [8]:
x0 = 1.0
h = 0.35
raw_w = np.exp(-0.5*((x-x0)/h)**2)
w = raw_w / raw_w.sum()

weight_df = pd.DataFrame({"x": x, "y": y, "weight": w})
weight_df = weight_df.sort_values("weight", ascending=False).head(12)

show_table(weight_df, f"Largest weights when predicting at x0={x0}", height=300)
print("Weighted prediction:", float(np.sum(w*y)))

x,y,weight
1.0363,0.6442,0.0677
1.0516,0.7848,0.0673
1.0550,0.7162,0.0672
1.0778,0.8834,0.0664
0.9207,1.0140,0.0663
1.0868,1.3068,0.0660
0.8622,0.1701,0.0630
0.8427,0.5686,0.0615
1.1680,0.9989,0.0606
1.1715,0.8889,0.0603


Weighted prediction: 0.8354621020415764


# Part VI — LOESS / local regression

LOESS is also local, but instead of merely averaging nearby $y$ values it fits a small polynomial inside each neighbourhood.

Conceptually:

1. choose a target $x_0$,
2. identify nearby observations,
3. assign distance-based weights,
4. fit a local regression,
5. evaluate that local regression at $x_0$,
6. move to the next $x_0$.

The `frac` parameter controls neighbourhood size.

In [56]:
p = figure(width=950, height=470,
           title="LOESS: local regressions across the input domain",
           x_axis_label="x", y_axis_label="y")
p.scatter(x, y, size=6, alpha=0.35, legend_label="observations")
p.line(grid, truth_grid, line_width=4, alpha=0.45, legend_label="true f(x)")

for frac,c in [(0.18,'red'), (0.35,'blue'), (0.65,'green')]:
    sm = lowess(y, x, frac=frac, it=0, return_sorted=True)
    p.line(sm[:,0], sm[:,1], line_width=2.5, legend_label=f"frac={frac}",color=c)

p.legend.location = "top_left"
p.legend.click_policy = "mute"
show(p)

# Part VII — Bias–variance experiment

Rather than discussing bias and variance abstractly, we can estimate them experimentally.

For each bandwidth:

1. repeatedly sample a new training set,
2. generate new observation noise,
3. fit the kernel estimator,
4. predict on a fixed grid,
5. compare the average fitted curve with the true curve.

At a fixed $x$,

$$
\operatorname{Bias}^2(x)
=
(E[\hat f(x)]-f(x))^2,
$$

and

$$
\operatorname{Var}(x)
=
E[(\hat f(x)-E[\hat f(x)])^2].
$$

In [50]:
def gaussian_nw(x_train, y_train, x_eval, h):
    d = (x_eval[:,None] - x_train[None,:]) / h
    W = np.exp(-0.5*d*d)
    W /= W.sum(axis=1, keepdims=True) + 1e-12
    return W @ y_train

eval_grid = np.linspace(-3, 3, 120)
eval_truth = f_true(eval_grid)

rows = []
for h in [0.08, 0.15, 0.30, 0.60, 1.00]:
    predictions = []
    for rep in range(60):
        rr = np.random.default_rng(1000+rep)
        xr = np.sort(rr.uniform(-3.2, 3.2, 70))
        yr = f_true(xr) + rr.normal(0, 0.35, 70)
        predictions.append(gaussian_nw(xr, yr, eval_grid, h))
    predictions = np.asarray(predictions)
    mean_pred = predictions.mean(axis=0)
    bias2 = np.mean((mean_pred-eval_truth)**2)
    variance = np.mean(predictions.var(axis=0, ddof=1))
    rows.append({
        "bandwidth": h,
        "bias_squared": bias2,
        "variance": variance,
        "bias2_plus_variance": bias2+variance,
    })

bv_df = pd.DataFrame(rows)
show_table(bv_df, "Empirical bias–variance decomposition", height=240)

p = figure(width=880, height=420, title="Bias–variance trade-off",
           x_axis_label="bandwidth", y_axis_label="average squared contribution")
for col, label,color in [
    ("bias_squared", "bias²","red"),
    ("variance", "variance","green"),
    ("bias2_plus_variance", "bias² + variance","blue"),
]:
    p.line(bv_df["bandwidth"], bv_df[col], line_width=3, legend_label=label,color=color)
    p.scatter(bv_df["bandwidth"], bv_df[col], size=7)
p.legend.location = "top_right"
show(p)

bandwidth,bias_squared,variance,bias2_plus_variance
0.08,0.0015,0.0496,0.0511
0.15,0.0018,0.0286,0.0304
0.30,0.0094,0.0188,0.0282
0.60,0.0715,0.0180,0.0896
1.00,0.2136,0.0188,0.2324


### Key inference

Very small bandwidth gives the model enough freedom to react to the exact noisy sample.  
That is why variance grows.

Very large bandwidth blends together observations from a broad region.  
That stabilises the estimator but washes out real curvature, increasing bias.

The best model is generally between the two extremes.

# Part VIII — Additive models on real regression data

A generalised additive modelling idea is

$$
g(E[Y\mid X])
=
\beta_0+\sum_{j=1}^{p}f_j(X_j).
$$

For ordinary regression with identity link,

$$
E[Y\mid X]
=
\beta_0+\sum_jf_j(X_j).
$$

This gives each predictor a nonlinear smooth contribution while keeping the overall model additive.

We build a GAM-like model using:

- separate spline bases produced by `SplineTransformer`,
- a ridge penalty to regularise the resulting coefficients.

This is not meant as a replacement for a dedicated GAM package; it is a transparent way to study the modelling principle.

In [36]:
diabetes = load_diabetes(as_frame=True)
X_reg = diabetes.data.copy()
y_reg = diabetes.target.copy()

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=RANDOM_STATE
)

linear_reg = LinearRegression()
additive_spline = Pipeline([
    ("spline", SplineTransformer(n_knots=5, degree=3, include_bias=False)),
    ("ridge", Ridge(alpha=3.0)),
])

rows = []
for name, model in [("Linear regression", linear_reg), ("Additive spline", additive_spline)]:
    model.fit(X_train_reg, y_train_reg)
    pred = model.predict(X_test_reg)
    rows.append({"model": name, **reg_metrics(y_test_reg, pred)})

show_table(pd.DataFrame(rows), "Linear baseline versus additive spline model", height=180)

model,RMSE,R2
Linear regression,53.3696,0.4849
Additive spline,51.3369,0.5234


### Visualising one fitted additive effect

We vary BMI while holding all other predictors at their training medians.

This asks:

> According to the fitted model, how does prediction change as BMI changes, conditional on a fixed reference profile?

This is a **model interpretation**, not a causal effect.

In [43]:
linear_reg.fit(X_train_reg, y_train_reg)
additive_spline.fit(X_train_reg, y_train_reg)

feature = "bmi"
values = np.linspace(
    X_train_reg[feature].quantile(0.01),
    X_train_reg[feature].quantile(0.99),
    180
)

base = X_train_reg.median().to_frame().T
scenario = pd.concat([base]*len(values), ignore_index=True)
scenario.columns = X_train_reg.columns
scenario[feature] = values

p = figure(width=900, height=420,
           title="Model-implied BMI contribution around a reference patient",
           x_axis_label="BMI feature", y_axis_label="predicted target")
p.line(values, linear_reg.predict(scenario), line_width=3, legend_label="linear",line_color=(200,45,0,0.6))
p.line(values, additive_spline.predict(scenario), line_width=3, legend_label="additive spline",line_color=(100,50,0,0.6))
p.legend.location = "top_left"
show(p)

# Part IX — Regression trees

A regression tree recursively partitions feature space using rules such as

$$
X_j<s.
$$

For a terminal region $R_m$, prediction is typically the regional mean:

$$
\hat y_{R_m}
=
\frac{1}{|R_m|}
\sum_{i:x_i\in R_m}y_i.
$$

Trees automatically represent:

- thresholds,
- nonlinearities,
- interactions.

But a fully grown tree can have high variance.

In [13]:
rows = []
for depth in [1, 2, 4, 8, None]:
    model = DecisionTreeRegressor(
        max_depth=depth,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train_reg, y_train_reg)
    train_pred = model.predict(X_train_reg)
    test_pred = model.predict(X_test_reg)
    rows.append({
        "max_depth": str(depth),
        "train_RMSE": mean_squared_error(y_train_reg, train_pred)**0.5,
        "test_RMSE": mean_squared_error(y_test_reg, test_pred)**0.5,
        "test_R2": r2_score(y_test_reg, test_pred),
    })

tree_depth_df = pd.DataFrame(rows)
show_table(tree_depth_df, "Tree depth: training fit versus test generalisation", height=250)

max_depth,train_RMSE,test_RMSE,test_R2
1,65.4899,65.8886,0.2149
2,58.2816,59.5463,0.3588
4,50.4348,58.6607,0.3777
8,31.3807,70.6455,0.0975
None,27.0899,72.6231,0.0462


## Cost-complexity pruning

A tree can first be grown and then pruned.

A common objective balances fit and tree size:

$$
R_\alpha(T)
=
R(T)+\alpha |T|,
$$

where:

- $R(T)$ measures training error,
- $|T|$ is a measure of tree complexity such as number of leaves,
- $\alpha$ penalises larger trees.

Larger $\alpha$ produces stronger pruning.

In [14]:
base_tree = DecisionTreeRegressor(random_state=RANDOM_STATE, min_samples_leaf=2)
path = base_tree.cost_complexity_pruning_path(X_train_reg, y_train_reg)
ccp_alphas = np.unique(path.ccp_alphas)

# Choose a compact spread of candidate alphas.
idx = np.linspace(0, len(ccp_alphas)-1, min(10, len(ccp_alphas))).astype(int)
alpha_grid = ccp_alphas[idx]

rows = []
for alpha in alpha_grid:
    model = DecisionTreeRegressor(
        random_state=RANDOM_STATE,
        min_samples_leaf=2,
        ccp_alpha=float(alpha),
    )
    model.fit(X_train_reg, y_train_reg)
    rows.append({
        "ccp_alpha": alpha,
        "n_leaves": model.get_n_leaves(),
        "train_RMSE": mean_squared_error(y_train_reg, model.predict(X_train_reg))**0.5,
        "test_RMSE": mean_squared_error(y_test_reg, model.predict(X_test_reg))**0.5,
    })

prune_df = pd.DataFrame(rows)
show_table(prune_df, "Cost-complexity pruning path", height=310)

ccp_alpha,n_leaves,train_RMSE,test_RMSE
0.0000,144,15.0744,77.0915
0.8225,131,15.2252,76.4561
1.9694,117,15.8401,76.3107
4.3151,102,17.2816,76.3409
7.7829,87,19.7001,75.7529
12.6469,70,23.5772,74.8057
18.3614,53,28.7473,72.4722
34.0803,38,34.6677,69.6738
55.6311,19,44.7727,62.1347
1755.6939,1,77.7472,74.8812


# Part X — Classification trees: Gini and entropy

For class probabilities $p_1,\ldots,p_K$:

### Gini impurity

$$
G=1-\sum_{k=1}^{K}p_k^2.
$$

### Entropy

$$
H=-\sum_{k=1}^{K}p_k\log p_k.
$$

Both are zero for a pure node.

In binary classification, impurity is highest near a 50/50 class mixture.

In [64]:
p1 = np.linspace(0.001, 0.999, 300)
gini = 2*p1*(1-p1)
entropy = -(p1*np.log2(p1)+(1-p1)*np.log2(1-p1))

p = figure(width=850, height=410, title="Binary-node impurity",
           x_axis_label="P(class=1)", y_axis_label="impurity")
p.line(p1, gini, line_width=3, legend_label="Gini")
p.line(p1, entropy, line_width=3, legend_label="Entropy",color=(25,10,0))
p.legend.location = "top_left"
show(p)

# Part XI — Bagging

Bagging means **bootstrap aggregating**.

Create bootstrap datasets

$$
D^{(1)},\ldots,D^{(B)},
$$

fit one unstable learner such as a deep tree to each, and average:

$$
\hat f_{\text{bag}}(x)
=
\frac{1}{B}\sum_{b=1}^{B}\hat f^{(b)}(x).
$$

Bagging mainly attacks **variance**.

The key point is not that every individual tree is excellent.  
It is that averaging many imperfect but not perfectly correlated trees stabilises prediction.

In [59]:
single_tree = DecisionTreeRegressor(min_samples_leaf=3, random_state=RANDOM_STATE)
bagging = BaggingRegressor(
    estimator=DecisionTreeRegressor(min_samples_leaf=3, random_state=RANDOM_STATE),
    n_estimators=250,
    bootstrap=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rows = []
for name, model in [("Single tree", single_tree), ("Bagged trees", bagging)]:
    model.fit(X_train_reg, y_train_reg)
    rows.append({
        "model": name,
        "train_RMSE": mean_squared_error(y_train_reg, model.predict(X_train_reg))**0.5,
        "test_RMSE": mean_squared_error(y_test_reg, model.predict(X_test_reg))**0.5,
        "test_R2": r2_score(y_test_reg, model.predict(X_test_reg)),
    })

show_table(pd.DataFrame(rows), "Single tree versus bagging", height=190)

model,train_RMSE,test_RMSE,test_R2
Single tree,27.0899,72.6231,0.0462
Bagged trees,30.8169,53.2667,0.4869


## Why 63.2% unique observations appear in a bootstrap sample

For one original observation, the probability of never being selected in $n$ bootstrap draws is

$$
\left(1-\frac{1}{n}\right)^n
\rightarrow e^{-1}\approx0.368.
$$

Therefore the probability of appearing at least once is

$$
1-e^{-1}\approx0.632.
$$

The remaining approximately 36.8% are **out-of-bag** for that bootstrap replicate.

In [60]:
n_boot = 500
fractions = []
for _ in range(1000):
    sample = rng.integers(0, n_boot, n_boot)
    fractions.append(len(np.unique(sample))/n_boot)

bootstrap_summary = pd.DataFrame({
    "quantity": [
        "theoretical unique fraction",
        "simulated unique fraction",
        "simulated OOB fraction",
    ],
    "value": [
        1-np.exp(-1),
        np.mean(fractions),
        1-np.mean(fractions),
    ],
})
show_table(bootstrap_summary, "Bootstrap 63.2% phenomenon", height=180)

quantity,value
theoretical unique fraction,0.6321
simulated unique fraction,0.6317
simulated OOB fraction,0.3683


# Part XII — Random forests

Bagged trees may still be highly correlated because the same very strong predictors can dominate many trees.

Random forests add feature subsampling:

> At each split, consider only a random subset of predictors.

This creates diversity.

Conceptually,

$$
\text{lower tree correlation}
\Rightarrow
\text{more useful averaging}
\Rightarrow
\text{lower ensemble variance}.
$$

In [61]:
rf_reg = RandomForestRegressor(
    n_estimators=400,
    max_features=0.7,
    min_samples_leaf=2,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf_reg.fit(X_train_reg, y_train_reg)

print("Test RMSE:", mean_squared_error(y_test_reg, rf_reg.predict(X_test_reg))**0.5)
print("Test R2:  ", r2_score(y_test_reg, rf_reg.predict(X_test_reg)))
print("OOB R2:   ", rf_reg.oob_score_)

Test RMSE: 53.90631777080777
Test R2:   0.4744929278478316
OOB R2:    0.4388242001522715


### OOB validation

Each training observation is excluded from some bootstrap samples.

We can predict an observation using only trees for which that observation was out-of-bag.

This provides a validation-like estimate without reserving an extra validation set for each tree.

OOB performance is not magical—it is another resampling estimate—but it is computationally convenient for bootstrap ensembles.

## Feature importance

Impurity-based importance asks:

> How much did splits on this feature reduce tree impurity across the ensemble?

Permutation importance asks a different question:

> How much does held-out predictive performance deteriorate when this feature's information is destroyed by shuffling?

Permutation importance is often easier to interpret as a predictive dependence measure, although correlated predictors can still complicate interpretation.

In [62]:
impurity = pd.DataFrame({
    "feature": X_reg.columns,
    "impurity_importance": rf_reg.feature_importances_,
})

perm = permutation_importance(
    rf_reg, X_test_reg, y_test_reg,
    scoring="neg_root_mean_squared_error",
    n_repeats=25,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_df = pd.DataFrame({
    "feature": X_reg.columns,
    "permutation_importance": perm.importances_mean,
    "permutation_sd": perm.importances_std,
})

importance_df = (
    impurity.merge(perm_df, on="feature")
    .sort_values("permutation_importance", ascending=False)
    .reset_index(drop=True)
)
show_table(importance_df, "Random Forest feature importance", height=310)

feature,impurity_importance,permutation_importance,permutation_sd
bmi,0.3188,10.7966,2.5365
s5,0.2293,10.3391,1.6318
bp,0.1204,2.0064,1.6353
s6,0.0658,0.9189,0.7380
sex,0.0108,0.8485,0.2272
s3,0.0606,0.5030,0.6328
s4,0.0345,0.3675,0.4166
age,0.0576,0.0756,0.6381
s2,0.0518,0.0121,0.3440
s1,0.0504,-0.8802,0.4521


# Part XIII — Boosting

Bagging builds trees largely independently.

Boosting builds an additive model **sequentially**:

$$
F_M(x)=F_0(x)+\sum_{m=1}^{M}\eta h_m(x).
$$

For squared-error regression, a useful intuition is that each new tree models remaining residual structure.

### Main complexity controls

- tree depth,
- number of boosting stages,
- learning rate $\eta$.

Learning rate and number of trees interact: smaller steps usually need more stages.

In [63]:
rows = []
for lr, n_estimators in [(0.02, 600), (0.05, 300), (0.10, 180), (0.30, 100)]:
    model = GradientBoostingRegressor(
        learning_rate=lr,
        n_estimators=n_estimators,
        max_depth=2,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train_reg, y_train_reg)
    pred = model.predict(X_test_reg)
    rows.append({
        "learning_rate": lr,
        "n_estimators": n_estimators,
        "test_RMSE": mean_squared_error(y_test_reg, pred)**0.5,
        "test_R2": r2_score(y_test_reg, pred),
    })

show_table(pd.DataFrame(rows), "Boosting learning-rate experiment", height=230)

learning_rate,n_estimators,test_RMSE,test_R2
0.02,600,55.9988,0.4329
0.05,300,57.1089,0.4102
0.10,180,57.4845,0.4024
0.30,100,60.1441,0.3458


# Part XIV — Support Vector Machines

## Linear maximum-margin classifier

A linear decision boundary is

$$
\beta_0+\beta^Tx=0.
$$

When classes are separable, the maximum-margin idea chooses the hyperplane with the greatest distance from the closest training observations.

Those closest observations are the **support vectors**.

They are important because changing them can move the boundary; many distant observations can move slightly without changing the optimal hyperplane.

In [21]:
X_blob, y_blob = make_blobs(
    n_samples=90,
    centers=[(-1.4,-1.2),(1.5,1.4)],
    cluster_std=[0.62,0.66],
    random_state=RANDOM_STATE,
)
blob_scaler = StandardScaler()
Xb = blob_scaler.fit_transform(X_blob)

svc = SVC(kernel="linear", C=1.0)
svc.fit(Xb, y_blob)

w = svc.coef_[0]
b = svc.intercept_[0]
xx = np.linspace(Xb[:,0].min()-0.4, Xb[:,0].max()+0.4, 250)
boundary = -(w[0]*xx+b)/w[1]
m1 = -(w[0]*xx+b-1)/w[1]
m2 = -(w[0]*xx+b+1)/w[1]

p = figure(width=850, height=480, title="Linear SVM: margin and support vectors",
           x_axis_label="standardised x1", y_axis_label="standardised x2")
p.scatter(Xb[y_blob==0,0], Xb[y_blob==0,1], size=7, alpha=0.6, legend_label="class 0")
p.scatter(Xb[y_blob==1,0], Xb[y_blob==1,1], size=7, alpha=0.6, marker="triangle", legend_label="class 1")
p.line(xx, boundary, line_width=3, legend_label="decision boundary")
p.line(xx, m1, line_width=2, line_dash="dashed", legend_label="margins")
p.line(xx, m2, line_width=2, line_dash="dashed")

sv = svc.support_vectors_
p.scatter(sv[:,0], sv[:,1], size=18, fill_alpha=0, line_width=3, legend_label="support vectors")
p.legend.location = "top_left"
show(p)

print("Number of support vectors:", len(svc.support_))

Number of support vectors: 5


## Soft margin and $C$

Real data are rarely perfectly separable.

A soft-margin SVM allows violations and trades them against margin size.

A common formulation is

$$
\min_{\beta,\beta_0,\xi}
\frac{1}{2}\|\beta\|^2+C\sum_i\xi_i.
$$

Interpretation:

- large $C$ → mistakes/violations are expensive → fit training data aggressively,
- small $C$ → violations are tolerated → stronger regularisation and wider margin.

Thus $C$ is another complexity knob.

In [22]:
rows = []
for C_value in [0.03, 0.3, 3, 100]:
    model = SVC(kernel="linear", C=C_value)
    model.fit(Xb, y_blob)
    rows.append({
        "C": C_value,
        "training_accuracy": model.score(Xb, y_blob),
        "n_support_vectors": len(model.support_),
        "margin_proxy_1_over_norm_w": 1/np.linalg.norm(model.coef_[0]),
    })
show_table(pd.DataFrame(rows), "Effect of C in a linear SVM", height=220)

C,training_accuracy,n_support_vectors,margin_proxy_1_over_norm_w
0.03,1.0,34,1.1833
0.30,1.0,8,0.7013
3.00,1.0,2,0.5414
100.00,1.0,2,0.5414


## Kernel trick and RBF SVM

A linear separator may be inadequate in the original feature space.

Kernel SVMs work with similarities

$$
K(x_i,x_j)=\langle\phi(x_i),\phi(x_j)\rangle
$$

without explicitly constructing the potentially high-dimensional mapping $\phi$.

The RBF kernel is

$$
K(x_i,x_j)
=
\exp(-\gamma\|x_i-x_j\|^2).
$$

- small $\gamma$ → broad influence and smoother boundary,
- large $\gamma$ → local influence and more intricate boundary.

This use of *kernel* differs from kernel regression: in SVMs the kernel acts as an inner-product/similarity mechanism.

In [23]:
rr = np.random.default_rng(123)
n_ring = 300
radii = np.concatenate([
    rr.normal(1.0, 0.18, n_ring//2),
    rr.normal(2.1, 0.22, n_ring//2),
])
angles = rr.uniform(0, 2*np.pi, n_ring)
X_ring = np.column_stack([radii*np.cos(angles), radii*np.sin(angles)])
y_ring = np.array([0]*(n_ring//2)+[1]*(n_ring//2))

sc = StandardScaler()
Xrs = sc.fit_transform(X_ring)

linear_ring = SVC(kernel="linear", C=2).fit(Xrs, y_ring)
rbf_ring = SVC(kernel="rbf", C=2, gamma=1.0).fit(Xrs, y_ring)

gx = np.linspace(Xrs[:,0].min()-0.4, Xrs[:,0].max()+0.4, 180)
gy = np.linspace(Xrs[:,1].min()-0.4, Xrs[:,1].max()+0.4, 180)
xxg, yyg = np.meshgrid(gx, gy)
mesh = np.column_stack([xxg.ravel(), yyg.ravel()])

def approximate_boundary(model):
    pred = model.predict(mesh).reshape(xxg.shape)
    cx = np.zeros_like(pred, dtype=bool)
    cy = np.zeros_like(pred, dtype=bool)
    cx[:,1:] = pred[:,1:] != pred[:,:-1]
    cy[1:,:] = pred[1:,:] != pred[:-1,:]
    mask = cx | cy
    return xxg[mask], yyg[mask]

bx1, by1 = approximate_boundary(linear_ring)
bx2, by2 = approximate_boundary(rbf_ring)

p1 = figure(width=430, height=420, title="Linear SVM")
p1.scatter(Xrs[y_ring==0,0], Xrs[y_ring==0,1], size=5, alpha=0.5)
p1.scatter(Xrs[y_ring==1,0], Xrs[y_ring==1,1], size=5, alpha=0.5, marker="triangle")
p1.scatter(bx1, by1, size=3)

p2 = figure(width=430, height=420, title="RBF SVM")
p2.scatter(Xrs[y_ring==0,0], Xrs[y_ring==0,1], size=5, alpha=0.5)
p2.scatter(Xrs[y_ring==1,0], Xrs[y_ring==1,1], size=5, alpha=0.5, marker="triangle")
p2.scatter(bx2, by2, size=3)

show(row(p1, p2))

# Part XV — Real classification case study

We now use the Breast Cancer Wisconsin dataset.

### Why pipelines matter

SVMs and neural networks are sensitive to scale.

If we standardise using the full dataset before cross-validation, validation-fold information leaks into preprocessing.

The correct pattern is therefore

$$
\text{scaler}
\rightarrow
\text{model}
$$

inside one scikit-learn `Pipeline`.

During cross-validation, each fold learns its own scaling parameters from the training portion only.

In [24]:
cancer = load_breast_cancer(as_frame=True)
X_cls = cancer.data.copy()
y_cls = cancer.target.copy()

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls,
    test_size=0.25,
    stratify=y_cls,
    random_state=RANDOM_STATE,
)

print("Shape:", X_cls.shape)
print("Target names:", list(cancer.target_names))
display(y_cls.value_counts().to_frame("count"))

Shape: (569, 30)
Target names: ['malignant', 'benign']


,count
target,
1,357
0,212


In [25]:
classification_models = {
    "Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=5, criterion="gini", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, max_features="sqrt", min_samples_leaf=2,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=180, learning_rate=0.05, max_depth=2, random_state=RANDOM_STATE
    ),
    "Linear SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="linear", C=1.0, probability=True, random_state=RANDOM_STATE)),
    ]),
    "RBF SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", C=2.0, gamma="scale", probability=True, random_state=RANDOM_STATE)),
    ]),
    "Neural Network": Pipeline([
        ("scale", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(40,20),
            alpha=1e-3,
            learning_rate_init=0.001,
            max_iter=1600,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )),
    ]),
}

rows = []
fitted_cls = {}
for name, model in classification_models.items():
    model.fit(X_train_cls, y_train_cls)
    fitted_cls[name] = model
    pred = model.predict(X_test_cls)
    prob = model.predict_proba(X_test_cls)[:,1]
    rows.append({"model": name, **cls_metrics(y_test_cls, pred, prob)})

cls_results = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
show_table(cls_results, "Held-out classification comparison", height=300)

model,Accuracy,Balanced_Accuracy,F1,ROC_AUC
RBF SVM,0.9860,0.9850,0.9889,0.9983
Logistic Regression,0.9860,0.9850,0.9889,0.9977
Linear SVM,0.9860,0.9850,0.9889,0.9973
Random Forest,0.9580,0.9512,0.9670,0.9939
Gradient Boosting,0.9510,0.9417,0.9617,0.9933
Neural Network,0.9301,0.9367,0.9425,0.9816
Decision Tree,0.9301,0.9251,0.9444,0.9314


## Why use several classification metrics?

### Accuracy

$$
\text{Accuracy}
=
\frac{\text{correct predictions}}{\text{all predictions}}.
$$

### Balanced accuracy

Average recall over classes. This is useful when classes are not equally represented.

### F1 score

$$
F_1
=
2\frac{\text{precision}\times\text{recall}}
{\text{precision}+\text{recall}}.
$$

### ROC AUC

Measures ranking discrimination over many thresholds.

No metric is universally best.  
The right metric depends on the downstream decision cost.

In [26]:
from bokeh.plotting import figure, show
from bokeh.models import Legend
from sklearn.metrics import roc_curve, roc_auc_score

p = figure(
    width=900,
    height=500,
    title="ROC curves on held-out data",
    x_axis_label="False Positive Rate",
    y_axis_label="True Positive Rate"
)

# Random baseline
p.line(
    [0, 1], [0, 1],
    line_dash="dashed",
    line_width=2,
    legend_label="random"
)

# Different dash patterns and markers
styles = [
    ("Logistic Regression", "yellow", "solid", "circle"),
    ("Random Forest", "red", "dashed", "square"),
    ("Gradient Boosting", "green", "dotdash", "triangle"),
    ("RBF SVM", "blue", "dotted", "diamond"),
    ("Neural Network", "purple", "dashdot", "inverted_triangle"),
]

for name, col, dash, marker in styles:
    model = fitted_cls[name]
    prob = model.predict_proba(X_test_cls)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_cls, prob)
    auc = roc_auc_score(y_test_cls, prob)

    p.line(
        fpr, tpr,
        line_width=2.5,
        color=col,
        line_dash=dash,
        legend_label=f"{name}: {auc:.3f}"
    )

    # Add markers every few points so the curve is easier to distinguish
    p.scatter(
        fpr[::2], tpr[::2],   # every 2nd point; adjust if too dense
        color=col,
        marker=marker,
        size=6,
        alpha=0.8
    )

p.legend.location = "bottom_right"
p.legend.click_policy = "hide"
show(p)

# Part XVI — Cross-validation

A single train/test split is itself random.

$K$-fold cross-validation repeatedly rotates the validation subset:

1. split data into $K$ folds,
2. train on $K-1$,
3. validate on the remaining fold,
4. repeat for every fold,
5. average the scores.

The mean estimates typical performance.

The fold-to-fold standard deviation helps reveal **stability**.

In [27]:
cv_cls = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, model in classification_models.items():
    scores = cross_validate(
        model, X_cls, y_cls,
        cv=cv_cls,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "f1": "f1",
            "roc_auc": "roc_auc",
        },
        n_jobs=-1,
    )
    rows.append({
        "model": name,
        "Accuracy_mean": scores["test_accuracy"].mean(),
        "Balanced_Accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "F1_mean": scores["test_f1"].mean(),
        "ROC_AUC_mean": scores["test_roc_auc"].mean(),
        "ROC_AUC_sd": scores["test_roc_auc"].std(ddof=1),
    })

cv_cls_df = pd.DataFrame(rows).sort_values("ROC_AUC_mean", ascending=False).reset_index(drop=True)
show_table(cv_cls_df, "5-fold cross-validated classification comparison", height=310)

model,Accuracy_mean,Balanced_Accuracy_mean,F1_mean,ROC_AUC_mean,ROC_AUC_sd
RBF SVM,0.9807,0.9770,0.9848,0.9959,0.0056
Logistic Regression,0.9737,0.9676,0.9794,0.9953,0.0060
Linear SVM,0.9737,0.9695,0.9793,0.9942,0.0096
Gradient Boosting,0.9490,0.9422,0.9599,0.9914,0.0057
Random Forest,0.9526,0.9480,0.9624,0.9898,0.0091
Neural Network,0.9491,0.9460,0.9590,0.9857,0.0132
Decision Tree,0.9262,0.9184,0.9420,0.9419,0.0135


### Why variability matters

Suppose one model has mean AUC 0.985 and another 0.982.

If fold-to-fold variability is around 0.02, the apparent 0.003 advantage is not strong evidence of a practically meaningful difference.

Model selection should consider both the mean and stability.

# Part XVII — SVM hyperparameter tuning

For an RBF SVM:

- $C$ controls the penalty for margin violations,
- $\gamma$ controls locality of the RBF kernel.

They interact.

A large $C$ plus large $\gamma$ can create a highly local, flexible decision rule.

We therefore tune them jointly using cross-validation.

In [28]:
svm_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)),
])

grid_search = GridSearchCV(
    svm_pipe,
    param_grid={
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": [0.001, 0.01, 0.1, "scale"],
    },
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
    return_train_score=True,
)

grid_search.fit(X_train_cls, y_train_cls)
print("Best parameters:", grid_search.best_params_)
print("Best CV AUC:", grid_search.best_score_)

tuning_df = pd.DataFrame(grid_search.cv_results_)[[
    "param_model__C",
    "param_model__gamma",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]].sort_values("rank_test_score").reset_index(drop=True)

show_table(tuning_df.head(12), "Top SVM hyperparameter settings", height=330)

Best parameters: {'model__C': 100, 'model__gamma': 0.001}
Best CV AUC: 0.9968283582089552


param_model__C,param_model__gamma,mean_train_score,mean_test_score,std_test_score,rank_test_score
100.0,0.001,0.9971,0.9968,0.0047,1
10.0,0.01,0.9981,0.9966,0.0048,2
10.0,0.001,0.9955,0.9948,0.0040,3
10.0,scale,1.0000,0.9946,0.0056,4
1.0,0.01,0.9956,0.9946,0.0038,5
1.0,scale,0.9977,0.9946,0.0036,6
100.0,0.01,1.0000,0.9940,0.0072,7
100.0,scale,1.0000,0.9907,0.0066,8
1.0,0.001,0.9919,0.9897,0.0065,9
1.0,0.1,0.9997,0.9888,0.0033,10


### Training score is diagnostically useful

When training score rises toward perfection while validation score stagnates or falls, the extra flexibility is not generalising.

This is a more informative view of tuning than simply printing the winning hyperparameters.

# Part XVIII — Neural networks

A hidden unit computes

$$
h_j
=
g\left(b_j+\sum_k w_{jk}x_k\right).
$$

For ReLU,

$$
g(z)=\max(0,z).
$$

A network composes nonlinear layers:

$$
X
\rightarrow H^{(1)}
\rightarrow H^{(2)}
\rightarrow\cdots\rightarrow Y.
$$

The hidden layers therefore learn a representation rather than relying on a manually chosen polynomial or spline basis.

## Gradient-based training

Collect all parameters into $\theta$ and minimise a loss:

$$
\min_\theta L(\theta).
$$

A generic gradient update is

$$
\theta_{t+1}
=
\theta_t-\eta\nabla_\theta L(\theta_t).
$$

Backpropagation computes these gradients efficiently by repeatedly applying the chain rule.

Neural-network optimisation is generally non-convex, so scaling, initialisation and optimisation settings matter.

In [34]:
scaler_nn = StandardScaler()
X_train_nn = scaler_nn.fit_transform(X_train_cls)

mlp_demo = MLPClassifier(
    hidden_layer_sizes=(50,25),
    alpha=1e-3,
    learning_rate_init=0.001,
    max_iter=1200,
    early_stopping=False,
    random_state=RANDOM_STATE,
)
mlp_demo.fit(X_train_nn, y_train_cls)

loss = mlp_demo.loss_curve_
p = figure(width=830, height=400, title="MLP optimisation: training loss",
           x_axis_label="iteration", y_axis_label="loss")
p.line(np.arange(1, len(loss)+1), loss, line_width=3)
show(p)

## $L_2$ weight decay

A regularised neural-network objective may look like

$$
L(\theta)+\lambda\|\theta\|_2^2.
$$

This is conceptually similar to ridge regression.

In scikit-learn's MLP, `alpha` controls the $L_2$ penalty strength.

Early stopping is another form of complexity control because it limits how far optimisation is allowed to adapt to training data.

In [35]:
rows = []
for alpha in [1e-6, 1e-4, 1e-2, 1.0]:
    model = Pipeline([
        ("scale", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=(40,20),
            alpha=alpha,
            learning_rate_init=0.001,
            max_iter=1500,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )),
    ])
    model.fit(X_train_cls, y_train_cls)
    train_prob = model.predict_proba(X_train_cls)[:,1]
    test_prob = model.predict_proba(X_test_cls)[:,1]
    rows.append({
        "alpha": alpha,
        "train_AUC": roc_auc_score(y_train_cls, train_prob),
        "test_AUC": roc_auc_score(y_test_cls, test_prob),
    })

show_table(pd.DataFrame(rows), "Neural-network regularisation experiment", height=220)

alpha,train_AUC,test_AUC
0.0000,0.9834,0.9816
0.0001,0.9834,0.9816
0.0100,0.9834,0.9816
1.0000,0.9834,0.9813


# Part XIX — Unified regression benchmark

Now compare several different representations of $f$ on the same regression split.

This table should not be read as a universal ranking of algorithms.  
A different dataset, tuning budget or sample size can reverse the ordering.

In [31]:
reg_models = {
    "Linear": LinearRegression(),
    "Additive splines": Pipeline([
        ("spline", SplineTransformer(n_knots=5, degree=3, include_bias=False)),
        ("ridge", Ridge(alpha=3.0)),
    ]),
    "Tree": DecisionTreeRegressor(max_depth=4, min_samples_leaf=5, random_state=RANDOM_STATE),
    "Bagging": BaggingRegressor(
        estimator=DecisionTreeRegressor(min_samples_leaf=3, random_state=RANDOM_STATE),
        n_estimators=250, n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=350, max_features=0.7, min_samples_leaf=2,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=2,
        min_samples_leaf=5, random_state=RANDOM_STATE
    ),
    "Neural Network": Pipeline([
        ("scale", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(40,20),
            alpha=1.0,
            learning_rate_init=0.005,
            max_iter=2000,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )),
    ]),
}

rows = []
for name, model in reg_models.items():
    model.fit(X_train_reg, y_train_reg)
    pred = model.predict(X_test_reg)
    rows.append({"model": name, **reg_metrics(y_test_reg, pred)})

reg_results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
show_table(reg_results, "Held-out regression benchmark", height=290)

model,RMSE,R2
Additive splines,51.3369,0.5234
Bagging,53.2667,0.4869
Linear,53.3696,0.4849
Random Forest,53.7968,0.4766
Tree,54.6489,0.4599
Gradient Boosting,57.1089,0.4102
Neural Network,58.8598,0.3735


# Part XX — Cross-validated regression comparison

A model that wins one split may not win consistently.

We therefore repeat evaluation across five folds and report both mean RMSE and its standard deviation.

In [32]:
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_subset = {
    "Linear": reg_models["Linear"],
    "Additive splines": reg_models["Additive splines"],
    "Tree": reg_models["Tree"],
    "Random Forest": reg_models["Random Forest"],
    "Gradient Boosting": reg_models["Gradient Boosting"],
}

rows = []
for name, model in cv_subset.items():
    scores = cross_validate(
        model, X_reg, y_reg,
        cv=cv_reg,
        scoring={"rmse": "neg_root_mean_squared_error", "r2": "r2"},
        n_jobs=-1,
    )
    rmse = -scores["test_rmse"]
    rows.append({
        "model": name,
        "CV_RMSE_mean": rmse.mean(),
        "CV_RMSE_sd": rmse.std(ddof=1),
        "CV_R2_mean": scores["test_r2"].mean(),
    })

cv_reg_df = pd.DataFrame(rows).sort_values("CV_RMSE_mean").reset_index(drop=True)
show_table(cv_reg_df, "5-fold cross-validated regression comparison", height=250)

model,CV_RMSE_mean,CV_RMSE_sd,CV_R2_mean
Additive splines,54.7256,3.3710,0.4827
Linear,54.8489,2.9528,0.4785
Gradient Boosting,57.0355,3.2184,0.4373
Random Forest,57.3011,3.6089,0.4329
Tree,61.6561,4.3295,0.3441


# Part XXI — A unifying representation map

The algorithms can be grouped by how they represent the unknown function.

### Global parametric

$$
f(x)=\beta_0+\beta^Tx.
$$

Example: linear regression.

### Basis expansion

$$
f(x)=\sum_j\beta_jb_j(x).
$$

Examples: polynomial regression and splines.

### Local modelling

$$
\hat f(x_0)
=
\text{function of observations near }x_0.
$$

Examples: kernel regression and LOESS.

### Partition-based modelling

$$
\mathcal X=R_1\cup\cdots\cup R_M.
$$

Example: decision trees.

### Ensemble modelling

$$
F(x)=\sum_m w_mh_m(x).
$$

Examples: bagging, random forests and boosting.

### Similarity-based margin modelling

$$
f(x)=\sum_i\alpha_iK(x_i,x)+b.
$$

Example: kernel SVM.

### Compositional representation learning

$$
f(x)=f_L(f_{L-1}(\cdots f_1(x))).
$$

Example: neural networks.

# Part XXII — One optimisation perspective

Many methods can be understood abstractly as

$$
\hat f
=
\arg\min_{f\in\mathcal F}
\left[
L(Y,f(X))
+
\lambda\Omega(f)
\right].
$$

- $\mathcal F$: allowed function family,
- $L$: data-fit loss,
- $\Omega(f)$: complexity penalty,
- $\lambda$: regularisation strength.

Examples:

### Ridge regression

$$
\sum_i(y_i-x_i^T\beta)^2+\lambda\|\beta\|_2^2.
$$

### Smoothing spline

$$
\sum_i(y_i-f(x_i))^2+\lambda\int[f''(t)]^2dt.
$$

### Soft-margin SVM

Margin size competes with a penalty for violations.

### Neural network

$$
L(\theta)+\lambda\|\theta\|_2^2.
$$

The mathematics differs, but the statistical theme is the same: **fit versus complexity**.

# Part XXIII — Hyperparameter map

| Method | Important knob | Intuition |
|---|---|---|
| Polynomial regression | degree | global curvature |
| Regression spline | knot count / regularisation | local flexibility |
| Smoothing spline | smoothing strength | curvature penalty |
| Kernel regression | bandwidth | neighbourhood size |
| LOESS | `frac` | local neighbourhood size |
| Decision tree | depth / leaf size / `ccp_alpha` | partition complexity |
| Random Forest | `max_features`, leaf size, trees | strength versus correlation |
| Boosting | learning rate, depth, stages | correction size and total capacity |
| SVM | $C$, $\gamma$ | margin penalty and kernel locality |
| Neural network | width, depth, `alpha`, epochs | representation and regularisation |

Do not memorise these as disconnected knobs.

The common question is

$$
\boxed{
\text{How complex should the learned function be?}
}
$$

# Part XXIV — Recommended workflow for an ST4248-style case study

## 1. Define the prediction problem

Specify target, observation unit and evaluation metric.

## 2. Search for leakage

Never allow future/post-outcome information into model training.

## 3. Establish a simple baseline

Use linear or logistic regression.

## 4. Diagnose nonlinearity

Use visualisation, residuals and domain knowledge.

## 5. Add flexibility deliberately

Try splines, local methods, trees, ensembles, kernels or neural networks because of a modelling reason—not because they are fashionable.

## 6. Tune inside cross-validation

Preprocessing must also remain inside the pipeline.

## 7. Preserve a final test set

Do not repeatedly tune against it.

## 8. Compare multiple metrics

Metric choice should reflect the decision problem.

## 9. Examine variability

A mean score without uncertainty can be misleading.

## 10. Interpret the fitted model

Inspect smooth components, support vectors, importance measures or error slices.

## 11. Prefer justified complexity

A tiny performance gain may not compensate for major losses in interpretability, speed or stability.

# Part XXV — Exercises

### Exercise 1 — Polynomial variance

Repeat polynomial degrees $1,3,5,10,20$ under five different random seeds.

**Question:** Which degree is most sensitive to the exact sample?

---

### Exercise 2 — Splines

Fix knot count and vary the ridge penalty.

**Question:** How can regularisation make a large basis behave smoothly?

---

### Exercise 3 — Kernel bandwidth

Create training/test error curves versus bandwidth.

**Question:** Why does a very small bandwidth usually have low training error but unstable test performance?

---

### Exercise 4 — LOESS

Compare `frac=0.1`, `0.3`, `0.6`, `0.9`.

**Question:** Relate neighbourhood size to effective degrees of freedom.

---

### Exercise 5 — Tree depth

Vary depth from 1 to 15 and plot train/test RMSE.

**Question:** Why can training error keep falling after test error starts rising?

---

### Exercise 6 — Pruning

Use cross-validation to select `ccp_alpha`.

**Question:** Why can a smaller pruned tree outperform the original full tree?

---

### Exercise 7 — Bagging

Record predictions of 50 bootstrap trees for one test observation.

**Question:** What quantity becomes more stable after averaging?

---

### Exercise 8 — Random Forest

Compare several `max_features` settings.

**Question:** Why can deliberately weakening individual trees improve the ensemble?

---

### Exercise 9 — Boosting

Create a grid over learning rate and number of stages.

**Question:** Why should these parameters be tuned jointly?

---

### Exercise 10 — SVM

Move one support vector and one non-support-vector in the toy dataset.

**Question:** Which move changes the boundary more?

---

### Exercise 11 — RBF SVM

Compare extreme values of $C$ and $\gamma$.

**Question:** Identify underfitting and overfitting from train/validation scores.

---

### Exercise 12 — Neural network

Compare a small network, a large network, strong $L_2$ penalty and early stopping.

**Question:** Why can the smaller model generalise better on a modest dataset?

---

### Exercise 13 — Model selection

Suppose boosting improves RMSE from 54 to 52 over an interpretable additive spline model.

Discuss whether you would deploy boosting when:

- explanations are legally required,
- latency is constrained,
- the 2-point RMSE improvement is stable,
- or the improvement disappears under cross-validation.

# Final synthesis

The surface-level story of ST4248 is a list of algorithms.

The deeper story is a sequence of increasingly flexible ways to represent $f$:

$$
\text{linear}
\rightarrow
\text{basis expansion}
\rightarrow
\text{local smoothing}
\rightarrow
\text{recursive partitioning}
\rightarrow
\text{ensembles}
\rightarrow
\text{kernel margins}
\rightarrow
\text{learned representations}.
$$

Across all of them, the same statistical questions keep returning:

1. What assumptions define the function class?
2. Which parameter controls flexibility?
3. What happens to bias and variance?
4. How should tuning be performed without leakage?
5. How stable is out-of-sample performance?
6. What interpretability is retained or lost?
7. Is the extra complexity actually justified?

The central lesson is:

$$
\boxed{
\text{Learn enough structure to predict well, but not enough noise to fool yourself.}
}
$$